# Module 1 — Brief 1 — Fine-tuning initial et évaluation

Ce notebook couvre les étapes du Brief 1 :

- Exploration du dataset FastIA
- Construction du prompt template
- Tokenisation et split train/test
- Chargement du modèle Llama 3.2 1B
- Configuration LoRA avec PEFT
- Entraînement avec tracking MLflow
- Evaluation quantitative et qualitative
- Diagnostic

Les cellules marquées `### A COMPLÉTER ###` contiennent du code partiel à terminer.
Les cellules sans marqueur sont fournies et peuvent être exécutées directement.

Le développement de l'API FastAPI (Brief 2) est hors de ce notebook.

---
## 0. Installation des dépendances

In [ ]:
# Exécuter cette cellule une seule fois
!pip install transformers peft datasets torch mlflow accelerate bitsandbytes

---
## 1. Imports et configuration

In [ ]:
import json
import torch
import mlflow
from collections import Counter
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType

print("Imports OK")

In [ ]:
# Détection automatique du device disponible
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device utilisé : {device}")

---
## 2. Exploration du dataset

Avant d'entraîner quoi que ce soit, il est indispensable de comprendre les données.
Un modèle ne peut pas être meilleur que les données sur lesquelles il est entraîné.

Cette étape permet de détecter des déséquilibres de classes, des anomalies ou des
caractéristiques qui influenceront les choix d'entraînement.

In [ ]:
# Chargement du dataset
with open("dataset_fastia_module1.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Nombre d'exemples : {len(raw_data)}")
print("\nExemple :")
print(json.dumps(raw_data[0], ensure_ascii=False, indent=2))

In [ ]:
### A COMPLÉTER ###
# Objectif : analyser la distribution des catégories
#
# 1. Construire la liste de toutes les catégories présentes dans le dataset
# 2. Compter le nombre d'exemples par catégorie avec Counter
# 3. Afficher le résultat

categories = [exemple["output"]["categorie"] for exemple in raw_data]
distribution_categories = # A COMPLÉTER

print("Distribution des catégories :")
# A COMPLÉTER

In [ ]:
### A COMPLÉTER ###
# Objectif : analyser la distribution des priorités
#
# 1. Construire la liste de toutes les priorités
# 2. Compter le nombre d'exemples par priorité
# 3. Afficher le résultat

priorites = # A COMPLÉTER
distribution_priorites = # A COMPLÉTER

print("Distribution des priorités :")
# A COMPLÉTER

In [ ]:
### A COMPLÉTER ###
# Objectif : analyser la longueur des demandes en entrée
#
# 1. Calculer la longueur en caractères de chaque champ 'input'
# 2. Calculer la longueur minimale, maximale et moyenne
# 3. Afficher les résultats
#
# Question : pourquoi la longueur des inputs est-elle importante
# pour le choix de MAX_LENGTH lors de la tokenisation ?
# Répondez dans un commentaire.

longueurs = # A COMPLÉTER

print(f"Longueur minimale  : {# A COMPLÉTER} caractères")
print(f"Longueur maximale  : {# A COMPLÉTER} caractères")
print(f"Longueur moyenne   : {# A COMPLÉTER:.0f} caractères")

# Votre réponse :
# ...

---
## 3. Construction du prompt template

Un modèle de langage apprend à partir de texte. Pour lui apprendre à classifier
et générer une réponse structurée en JSON, chaque exemple du dataset doit être
transformé en une paire instruction / réponse dans un format cohérent.

Ce format s'appelle un prompt template. Llama 3.2 utilise le format suivant :

```
<s>[INST] instruction [/INST] réponse </s>
```

Le modèle apprend à reproduire ce pattern : quand il voit [INST]...[/INST],
il génère la suite.

In [ ]:
def build_prompt(exemple):
    """
    Transforme un exemple du dataset en prompt d'instruction.

    Args:
        exemple (dict): un exemple avec les clés 'input' et 'output'

    Returns:
        str: le prompt formaté
    """
    instruction = (
        "Tu es un assistant FastIA. "
        "Analyse la demande suivante et réponds uniquement en JSON valide "
        "avec les champs : categorie, priorite, reponse_suggeree.\n\n"
        f"Demande : {exemple['input']}"
    )
    reponse = json.dumps(exemple["output"], ensure_ascii=False)

    return f"<s>[INST] {instruction} [/INST] {reponse} </s>"


# Test sur le premier exemple
print(build_prompt(raw_data[0]))

In [ ]:
### A COMPLÉTER ###
# Objectif : appliquer build_prompt à tous les exemples du dataset
#
# 1. Créer une liste 'prompts' contenant le prompt formaté de chaque exemple
# 2. Afficher la longueur de cette liste
# 3. Afficher le prompt du 5ème exemple pour vérification

prompts = # A COMPLÉTER

print(f"Nombre de prompts : {len(prompts)}")
print("\nExemple (index 4) :")
print(# A COMPLÉTER)

---
## 4. Tokenisation et split train/test

Les modèles de langage ne travaillent pas avec du texte brut mais avec des tokens —
des unités de texte représentées par des entiers.

Le tokenizer est spécifique à chaque modèle. Il faut toujours utiliser le tokenizer
correspondant au modèle de base, jamais un tokenizer générique.

In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-1B"

# Chargement du tokenizer
# Un token d'accès HuggingFace est nécessaire pour ce modèle.
# Connectez-vous sur huggingface.co et acceptez les conditions d'utilisation Meta.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print(f"Taille du vocabulaire : {tokenizer.vocab_size} tokens")
print(f"Token de padding : {tokenizer.pad_token}")

In [ ]:
MAX_LENGTH = 512

def tokenize(prompt):
    """
    Tokenise un prompt et prépare les labels pour l'entraînement.

    Args:
        prompt (str): le prompt formaté

    Returns:
        dict: input_ids, attention_mask, labels
    """
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result


# Test sur un exemple
exemple_tokenise = tokenize(prompts[0])
print(f"Longueur tokenisée : {len(exemple_tokenise['input_ids'])} tokens")

In [ ]:
### A COMPLÉTER ###
# Objectif : construire les jeux d'entraînement et de test
#
# 1. Créer un Dataset HuggingFace à partir de la liste 'prompts'
#    Indice : Dataset.from_dict({"text": prompts})
#
# 2. Tokeniser tous les exemples avec .map()
#    Indice : dataset.map(lambda x: tokenize(x["text"]))
#
# 3. Diviser en train (80%) et test (20%) avec .train_test_split()
#
# 4. Afficher la taille des deux jeux
#
# Question : pourquoi conserver un jeu de test séparé du jeu d'entraînement ?
# Répondez dans un commentaire.

dataset = Dataset.from_dict({"text": prompts})
dataset_tokenise = # A COMPLÉTER
split = # A COMPLÉTER

train_dataset = split["train"]
test_dataset = split["test"]

print(f"Entraînement : {len(train_dataset)} exemples")
print(f"Test         : {len(test_dataset)} exemples")

# Votre réponse :
# ...

---
## 5. Chargement du modèle de base

On charge Llama 3.2 1B sans modifier ses poids. A ce stade, c'est le modèle
générique pré-entraîné par Meta sur des milliards de tokens.

L'étape suivante (LoRA) va ajouter des couches adaptateurs légères sur ce modèle
sans toucher aux poids originaux.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres totaux : {total_params:,}")

---
## 6. Configuration LoRA

LoRA (Low-Rank Adaptation) est une technique de fine-tuning efficace : plutôt que
de modifier tous les poids du modèle, on ajoute de petites matrices d'adaptation
sur certaines couches.

Résultat : on entraîne seulement environ 1% des paramètres du modèle.

Paramètres clés :
- r : rang des matrices — plus élevé = plus de capacité, plus de mémoire
- lora_alpha : facteur d'échelle — généralement égal à 2 * r
- target_modules : couches sur lesquelles appliquer LoRA
- lora_dropout : régularisation pour éviter le sur-apprentissage

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
### A COMPLÉTER ###
# Objectif : calculer et afficher le pourcentage de paramètres entraînables
#
# 1. Calculer le nombre de paramètres entraînables (requires_grad == True)
# 2. Calculer le total des paramètres
# 3. Calculer et afficher le pourcentage
#
# Question : quel est l'avantage de n'entraîner qu'une fraction des paramètres
# sur un hardware avec une mémoire limitée ?
# Répondez dans un commentaire.

trainable = # A COMPLÉTER
total = # A COMPLÉTER
pct = # A COMPLÉTER

print(f"Paramètres entraînables : {trainable:,} ({pct:.2f}% du total)")

# Votre réponse :
# ...

---
## 7. Entraînement

On lance le run d'entraînement. Chaque run est tracké dans MLflow pour conserver
une trace des paramètres et des métriques.

Hyperparamètres à retenir :
- learning_rate : vitesse d'apprentissage
- num_train_epochs : nombre de passages sur le dataset complet
- per_device_train_batch_size : nombre d'exemples traités simultanément

In [ ]:
mlflow.set_experiment("fastia-llama-finetuning")

TRAINING_CONFIG = {
    "learning_rate": 2e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "run_name": "run1-lr2e4-3epochs"
}

with mlflow.start_run(run_name=TRAINING_CONFIG["run_name"]):

    mlflow.log_params({
        "model_id": MODEL_ID,
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "learning_rate": TRAINING_CONFIG["learning_rate"],
        "num_train_epochs": TRAINING_CONFIG["num_train_epochs"],
        "batch_size": TRAINING_CONFIG["per_device_train_batch_size"],
        "dataset_size": len(raw_data),
        "max_length": MAX_LENGTH
    })

    training_args = TrainingArguments(
        output_dir="./model_run1",
        num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
        per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        logging_steps=10,
        save_strategy="epoch",
        evaluation_strategy="epoch",
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        tokenizer=tokenizer
    )

    result = trainer.train()

    mlflow.log_metric("train_loss", result.training_loss)
    eval_result = trainer.evaluate()
    mlflow.log_metric("eval_loss", eval_result["eval_loss"])

    print(f"Train loss : {result.training_loss:.4f}")
    print(f"Eval loss  : {eval_result['eval_loss']:.4f}")

---
## 8. Evaluation quantitative

La loss mesure l'erreur du modèle sur les données. Elle ne dit pas directement
si les prédictions sont correctes, mais elle indique si le modèle apprend.

Deux signaux importants à analyser :
- La courbe de loss : descend-elle régulièrement ?
- L'écart train loss / eval loss : un écart important signale un sur-apprentissage.

In [ ]:
### A COMPLÉTER ###
# Objectif : récupérer et afficher les métriques du run depuis MLflow
#
# 1. Récupérer les runs de l'expérience avec mlflow.search_runs()
#    Indice : mlflow.search_runs(experiment_names=["fastia-llama-finetuning"])
#
# 2. Afficher les colonnes suivantes pour le run :
#    - run_name
#    - learning_rate
#    - num_train_epochs
#    - train_loss
#    - eval_loss
#
# Question : que vous indique l'écart entre train_loss et eval_loss ?
# Répondez dans un commentaire.

runs = mlflow.search_runs(experiment_names=["fastia-llama-finetuning"])
# A COMPLÉTER

# Votre réponse :
# ...

---
## 9. Evaluation qualitative

La loss est une métrique indirecte. Il faut aussi vérifier que le modèle produit
des sorties réellement utilisables : JSON valide, catégories reconnues, réponses
cohérentes avec la demande.

In [ ]:
def predict(texte, model, tokenizer, max_new_tokens=150):
    """
    Génère une prédiction à partir d'un texte de demande.

    Args:
        texte (str): la demande client en texte libre
        model: le modèle fine-tuné
        tokenizer: le tokenizer associé
        max_new_tokens (int): nombre maximum de tokens à générer

    Returns:
        dict: la réponse parsée en JSON, ou un dict d'erreur si le parsing échoue
    """
    prompt = (
        f"<s>[INST] Tu es un assistant FastIA. "
        f"Analyse la demande suivante et réponds uniquement en JSON valide "
        f"avec les champs : categorie, priorite, reponse_suggeree.\n\n"
        f"Demande : {texte} [/INST]"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    try:
        return json.loads(generated)
    except json.JSONDecodeError:
        return {"erreur": "JSON invalide", "brut": generated}


# Test rapide
test_input = "Notre serveur est tombé, personne ne peut travailler."
print(json.dumps(predict(test_input, model, tokenizer), ensure_ascii=False, indent=2))

In [ ]:
### A COMPLÉTER ###
# Objectif : évaluer le modèle sur 10 demandes couvrant les 5 catégories
#
# 1. Choisir 10 demandes (au moins une par catégorie)
#    Les demandes ne doivent pas être copiées du dataset d'entraînement
#
# 2. Pour chaque demande, appeler predict() et vérifier :
#    - Le JSON est-il valide ? (pas de clé 'erreur' dans le résultat)
#    - La catégorie est-elle dans CATEGORIES_VALIDES ?
#    - La priorité est-elle 'haute' ou 'normale' ?
#
# 3. Afficher un tableau récapitulatif :
#    Demande | Catégorie prédite | Priorité | JSON valide

CATEGORIES_VALIDES = [
    "Support technique",
    "Demande commerciale",
    "Demande de transformation",
    "Réclamation",
    "Information générale"
]

demandes_test = [
    # A COMPLÉTER : 10 demandes de votre choix
]

resultats = []
for demande in demandes_test:
    prediction = predict(demande, model, tokenizer)
    # A COMPLÉTER : construire et afficher le tableau récapitulatif

---
## 10. Diagnostic

Le diagnostic est une étape de réflexion structurée sur les résultats obtenus.
Il conditionne les actions correctives du Brief 2.

Répondez aux questions suivantes dans les cellules ci-dessous.
Appuyez-vous sur les métriques MLflow et sur les résultats qualitatifs.

In [ ]:
### A COMPLÉTER ###
# Question 1 : classification
#
# Le modèle classifie-t-il correctement toutes les catégories ?
# Certaines catégories sont-elles mieux gérées que d'autres ?
# Les erreurs sont-elles aléatoires ou systématiques ?

# Votre analyse :
# ...

In [ ]:
### A COMPLÉTER ###
# Question 2 : lecture de la loss
#
# Que vous indique l'écart entre train_loss et eval_loss ?
# Le modèle est-il en sous-apprentissage ou en sur-apprentissage ?
# Justifiez à partir des valeurs observées.

# Votre analyse :
# ...

In [ ]:
### A COMPLÉTER ###
# Question 3 : qualité de la sortie JSON
#
# Le modèle génère-t-il systématiquement du JSON valide ?
# Si non, dans quels cas le JSON est-il invalide ou mal formé ?

# Votre analyse :
# ...

In [ ]:
### A COMPLÉTER ###
# Question 4 : actions correctives
#
# Sur la base de votre diagnostic, proposez les actions correctives
# que vous mettrez en oeuvre dans le Brief 2.
#
# Pour chaque action, précisez :
# - Ce que vous allez faire (compléter le dataset, ajuster les hyperparamètres, les deux)
# - Pourquoi cette action devrait améliorer les résultats
# - Ce que vous espérez observer sur la loss et les prédictions

# Vos propositions :
# ...

---
## Récapitulatif

Etapes réalisées dans ce notebook :

1. Exploration du dataset : distribution des catégories, priorités, longueurs
2. Construction du prompt template d'instruction
3. Tokenisation et split train/test
4. Chargement de Llama 3.2 1B
5. Configuration et application de LoRA
6. Entraînement tracké dans MLflow
7. Evaluation quantitative : analyse de la loss
8. Evaluation qualitative : 10 prédictions sur cas réels
9. Diagnostic et propositions d'actions correctives

Prochaine étape — Brief 2 : mettre en oeuvre les actions correctives,
réentraîner, comparer les runs et exposer le meilleur modèle via FastAPI.